In [116]:
import pandas as pd

In [117]:
df = pd.read_csv("quikr_car.csv")
df.head()

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing XO eRLX Euro III,Hyundai,2007,"80,000","45,000 kms",Petrol
1,Mahindra Jeep CL550 MDI,Mahindra,2006,"4,25,000",40 kms,Diesel
2,Maruti Suzuki Alto 800 Vxi,Maruti,2018,Ask For Price,"22,000 kms",Petrol
3,Hyundai Grand i10 Magna 1.2 Kappa VTVT,Hyundai,2014,"3,25,000","28,000 kms",Petrol
4,Ford EcoSport Titanium 1.5L TDCi,Ford,2014,"5,75,000","36,000 kms",Diesel


In [118]:
backup = df.copy()

# Data Problems

- remove non year values
- year object into integer
- price has ask for price value
- price object into integer without comma
- kms_driven has non integer values
- kms_driven object into integer without kms and comma
- fuel_type has nan values
- first three words in name 

# Solutions

In [119]:
df = df[df['year'].str.isnumeric()]

In [120]:
df['year'] = df['year'].astype(int)

In [121]:
df = df[df['Price']!="Ask For Price"]

In [122]:
df['Price'] = df['Price'].str.replace(",","").astype(int)

In [123]:
df["kms_driven"] = df["kms_driven"].str.split(" ").str.get(0).str.replace(",", "")

In [124]:
df = df[df["kms_driven"].str.isnumeric()]

In [125]:
df['kms_driven'] = df["kms_driven"].astype(int)

In [126]:
df = df[~df["fuel_type"].isna()]

In [127]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 816 entries, 0 to 889
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        816 non-null    object
 1   company     816 non-null    object
 2   year        816 non-null    int64 
 3   Price       816 non-null    int64 
 4   kms_driven  816 non-null    int64 
 5   fuel_type   816 non-null    object
dtypes: int64(3), object(3)
memory usage: 44.6+ KB


In [132]:
df['name'] = df['name'].str.split(' ').str.slice(0,3).str.join(' ')

In [141]:
df = df.reset_index(drop=True)
df

,name,company,year,Price,kms_driven,fuel_type
0,Hyundai Santro Xing,Hyundai,2007,80000,45000,Petrol
1,Mahindra Jeep CL550,Mahindra,2006,425000,40,Diesel
2,Hyundai Grand i10,Hyundai,2014,325000,28000,Petrol
3,Ford EcoSport Titanium,Ford,2014,575000,36000,Diesel
4,Ford Figo,Ford,2012,175000,41000,Diesel
...,...,...,...,...,...,...
811,Maruti Suzuki Ritz,Maruti,2011,270000,50000,Petrol
812,Tata Indica V2,Tata,2009,110000,30000,Diesel
813,Toyota Corolla Altis,Toyota,2009,300000,132000,Petrol
814,Tata Zest XM,Tata,2018,260000,27000,Diesel


In [142]:
df.describe()

,year,Price,kms_driven
count,816.000000,8.160000e+02,816.000000
mean,2012.444853,4.117176e+05,46275.531863
std,4.002992,4.751844e+05,34297.428044
min,1995.000000,3.000000e+04,0.000000
25%,2010.000000,1.750000e+05,27000.000000
50%,2013.000000,2.999990e+05,41000.000000
75%,2015.000000,4.912500e+05,56818.500000
max,2019.000000,8.500003e+06,400000.000000


In [147]:
df = df[df['Price']< 6000000].reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 815 entries, 0 to 814
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   name        815 non-null    object
 1   company     815 non-null    object
 2   year        815 non-null    int64 
 3   Price       815 non-null    int64 
 4   kms_driven  815 non-null    int64 
 5   fuel_type   815 non-null    object
dtypes: int64(3), object(3)
memory usage: 38.3+ KB


In [149]:
df.to_csv("cleaned car data.csv")

# Extracting Training Data

In [152]:
from sklearn.model_selection import train_test_split

In [153]:
X = df[["name", "company", "year", "kms_driven", "fuel_type"]]
Y = df["Price"]

In [154]:
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size = 0.2)

In [155]:
from sklearn.linear_model import LinearRegression

In [157]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

## Creating One Hot Encoder Object to contain all categorical values

In [158]:
model = LinearRegression()

In [161]:
encoder = OneHotEncoder()
encoder.fit(X[["name", "company", "fuel_type"]])

,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


In [166]:
column_trans = make_column_transformer((OneHotEncoder(categories = encoder.categories_), ["name", "company", "fuel_type"]), remainder="passthrough")

In [170]:
pipe = make_pipeline(column_trans, model)
pipe.fit(X_train, y_train)

,steps,"[('columntransformer', ...), ('linearregression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('onehotencoder', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [172]:
y_pred = pipe.predict(X_test)
y_pred

array([-134249.81268708,  360556.4950752 ,  465162.10354355,
        299302.24995658,  386044.20070225,  210936.41172695,
         83345.2615049 ,  159186.29560699,  -24234.20661582,
        752094.86916548,  676998.13946374,  264117.3462266 ,
        595541.76907498,  382671.60276043,  214926.21647012,
        619298.70610173,  202313.00960346,  452447.34221762,
         43558.27268489,  595112.56727116,  177956.60434417,
        745514.70597599,  563708.22293454,  303045.5925951 ,
        619885.64464696,  -99648.57165711, 1089187.99841094,
        741306.878493  ,  241150.11667553,  150987.06332219,
        182552.14416219,  467480.59030972,  253491.38180003,
        524099.49696411,  303183.99164124,   22591.89342354,
        467552.15641072,  318067.92079817,  199802.08396512,
        249230.8842001 ,  414078.59748021,  338668.49542117,
        376728.67049977,  277792.2972957 ,  338424.05811538,
        179580.92963666,  207506.42287812,  117336.91623288,
        450075.64522961,

In [173]:
r2_score(y_test, y_pred)

0.4364016379329676

# Finding the model with random state which has highest r2 score

In [178]:
scores = []
for i in range(1000):
    X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size = 0.2, random_state=i)
    model = LinearRegression()
    pipe = make_pipeline(column_trans, model)
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    scores.append(r2_score(y_test, y_pred))

In [181]:
import numpy as np
np.argmax(scores)

np.int64(433)

In [193]:
pipe.predict(pd.DataFrame(columns = X_test.columns, data = np.array(["Maruti Suzuki Swift", "Maruti", 2018, 20000, "Diesel"]).reshape(1,5)))

array([478500.13885742])

In [187]:
scores[np.argmax(scores)]

0.8456642667912282

In [195]:
import pickle

pickle.dump(pipe, open("LinearModelPrediction.pkl", "wb"))